# M8 on Kaggle: full-pool FedAvg and FedProx (E2)

**Seed:** set `SEED` in the first code cell. Run 42 first, then repeat with 43, as for M7. The partition stays the same, only the model seeds change.

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*. The MLP trains on CPU, as the README's M8 plan says.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed. An interactive session can stop when it is left alone.
Time is an estimate: about 1 to 2 hours for the six passes, so allow 3.

Each pass runs in its own process, and the zip is rewritten after each one, so a failure in one pass does not lose the others. Every pass prints an exit code: 137 means the process was killed (usually out of memory). `--report` then writes the report from whatever finished, and says which passes are missing.

**Output:** `/kaggle/working/m8_full_seed42_outputs.zip` (the name follows `SEED`) contains `reports/m8/full/` and the run folders. Download it and bring it back, also after a failure.


In [ ]:
SEED = 42
TAG = "" if SEED == 42 else f"seed{SEED}"   # seed 42 is the main report, as in M7
TAG_ARG = f"--tag {TAG}" if TAG else ""
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full


In [ ]:
import glob, os, shutil
REP = os.path.join("reports/m8/full", TAG)
def pack():
    stage = "/kaggle/working/m8_stage"
    shutil.rmtree(stage, ignore_errors=True)
    if os.path.isdir(REP):
        shutil.copytree(REP, f"{stage}/{REP}")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m8_fedavg_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive(f"/kaggle/working/m8_full_seed{SEED}_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])


In [ ]:
# All passes, each in its own process. 4 threads, because results reproduce exactly only at the same thread count
for name in ["fedavg", "fedavg_no_rehearsal", "fedprox_mu0.001", "fedprox_mu0.01", "fedprox_mu0.1", "fedprox_mu1"]:
    print("=====", name, flush=True)
    !python -X faulthandler scripts/m8_fedavg.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {TAG_ARG} --only {name} --resume; echo "exit code: $?"
    pack()


In [ ]:
# Report from whatever finished
!python -X faulthandler scripts/m8_fedavg.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {TAG_ARG} --report; echo "exit code: $?"
pack()
if os.path.exists(f"{REP}/FEDAVG.md"):
    print(open(f"{REP}/FEDAVG.md", encoding="utf-8").read()[:6000])
